# Classification Track (Part A) - Hotel Booking Demand

**Problem statement.** Predict `is_canceled` - whether a booking will
eventually be cancelled - from the attributes known at the time the booking
is made. A hotel that can flag a booking as high-cancellation-risk before
arrival can overbook more safely and target offers at the guests who are most likely to cancel.

**Dataset.** `data/hotel_bookings.csv`, 119,390 bookings x 32 columns, from
Antonio, Almeida & Nunes (2019), *Hotel booking demand datasets*,
Data in Brief 22:41-49. Same file as the regression track; different target.

**Scope of this notebook.** Review 1 covers Classification Part A: the first
five algorithms (Logistic Regression, KNN, Naive Bayes, Decision Tree, SVM).

**Notebook map**:

| Section | Contents | Rubric |
|---|---|---|
| 1 | Load & audit | A1 |
| 2 | Exploratory data analysis | A2, A3 |
| 3 | Cleaning | B1 |
| 4 | Feature engineering | B3 |
| 5 | Encoding, scaling & splitting | B2 |
| 6 | Dimensionality reduction - PCA vs LDA | - |
| 7 | Five classification algorithms | D1 |
| 8 | Comparative evaluation | D2 |
| 9 | Cross-validation | (7.2) |
| 10 | Conclusion | - |


In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

DATA = "../data/hotel_bookings.csv"


ModuleNotFoundError: No module named 'seaborn'

## 1. Load & audit  *(A1)*

In [ ]:
df_raw = pd.read_csv(DATA)
print(f"rows: {df_raw.shape[0]:,}    columns: {df_raw.shape[1]}")
df_raw.head()


In [ ]:
audit = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing": df_raw.isna().sum(),
    "missing_%": (df_raw.isna().mean() * 100).round(2),
    "unique": df_raw.nunique(),
})
audit


In [ ]:
print("Target `is_canceled` (0 = kept, 1 = cancelled)")
counts = df_raw["is_canceled"].value_counts()
rate = df_raw["is_canceled"].mean() * 100
print(counts)
print(f"\ncancellation rate: {rate:.1f}%")


**Observation.** No column is fully empty, but four have large gaps: `company` (94.3%), `agent`
(13.7%), `country` (0.4%) and `children` (4 rows) - handled individually in a later section. The target is imbalanced at roughly 37% cancelled against 63%
kept, not extreme, but weighted F1 rather than plain accuracy is used
throughout so the majority class does not dominate the score.

## 2. Exploratory data analysis  *(A2, A3)*

### 2.1 Target distribution

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df_raw, x="is_canceled", hue="is_canceled",
              palette=["#4C72B0", "#C44E52"], legend=False, ax=ax)
ax.set(title="Booking outcome", xlabel="is_canceled", ylabel="bookings")
ax.set_xticks([0, 1])
ax.set_xticklabels(["kept (0)", "cancelled (1)"])
for p in ax.patches:
    ax.annotate(f"{p.get_height():,}\n({p.get_height()/len(df_raw)*100:.1f}%)",
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
plt.tight_layout()
plt.show()


**Observation:** 63% of bookings are kept and 37% cancelled. That is
close enough to balanced that no resampling is
needed to get a usable model, but it is skewed enough that plain accuracy is
misleading - a model that always predicts "kept" already scores 63%. Weighted
F1 and the confusion matrix 8 are what actually judge these
models.

### 2.2 Distributions of the numeric features

In [ ]:
numeric_raw = df_raw.select_dtypes("number").columns.drop(["is_canceled"])
fig, axes = plt.subplots(5, 4, figsize=(15, 14))
for ax, col in zip(axes.ravel(), numeric_raw):
    sns.histplot(df_raw[col], bins=30, ax=ax, color="#4C72B0")
    ax.set(title=col, xlabel="", ylabel="")
for ax in axes.ravel()[len(numeric_raw):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()


**Observation:** Most numeric columns are heavily right-skewed or zero
inflated - `previous_cancellations`, `booking_changes`,
`days_in_waiting_list`, `babies`, `required_car_parking_spaces`.
`lead_time` in particular has a long tail out past 400 days; Section 2.4
checks whether that tail carries cancellation signal.

### 2.3 Correlation structure

In [ ]:
corr_cols = list(numeric_raw) + ["is_canceled"]
corr = df_raw[corr_cols].corr()
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, cmap="vlag", center=0, annot=False, square=True,
            linewidths=.4, cbar_kws={"label": "Pearson r"}, ax=ax)
ax.set_title("Correlation matrix, numeric columns + is_canceled")
plt.tight_layout()
plt.show()

print(corr["is_canceled"].drop("is_canceled").sort_values(key=abs, ascending=False).round(3))


**Observation.** `lead_time` is the strongest single numeric
correlate of cancellation (r ~ 0.29): bookings made far in advance are more
likely to fall through. `total_of_special_requests` and
`required_car_parking_spaces` both correlate negatively - guests who commit
to specifics tend to follow through. `previous_cancellations` correlates
positively but weakly on its own, which is why Section 4 combines it with
`previous_bookings_not_canceled` into a rate rather than using the raw
count.

### 2.4 Feature-target relationships

In [ ]:
sample = df_raw.sample(6000, random_state=RANDOM_STATE)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

sns.boxplot(data=sample, x="is_canceled", y="lead_time", ax=axes[0], color="#4C72B0")
axes[0].set(title="Lead time by outcome", xlabel="is_canceled", ylabel="lead time (days)")

sns.boxplot(data=sample, x="is_canceled", y="adr", ax=axes[1], color="#55A868")
axes[1].set(title="ADR by outcome", xlabel="is_canceled", ylabel="ADR (EUR)", ylim=(0, 300))

sns.countplot(data=sample, x="total_of_special_requests", hue="is_canceled",
              ax=axes[2])
axes[2].set(title="Special requests by outcome", xlabel="special requests", ylabel="bookings")
axes[2].legend(title="is_canceled", labels=["kept", "cancelled"])

plt.tight_layout()
plt.show()


**Observation.** Cancelled bookings have a visibly higher median lead
time and a biggerer upper whisker - the longer a booking sits on the books,
the more chances something changes. ADR barely separates the two groups,
consistent with Section 2.3's weak correlation. Special requests fall off
sharply for cancelled bookings: guests with zero special requests cancel
noticeably more often than guests with two or more, which can be interpreted as
engagement with the specific stay rather than a generic reservation.

### 2.5 Categorical drivers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

for ax, col in zip(axes, ["deposit_type", "market_segment", "customer_type"]):
    rate = df_raw.groupby(col)["is_canceled"].mean().sort_values(ascending=False) * 100
    sns.barplot(x=rate.values, y=rate.index, ax=ax, color="#4C72B0")
    ax.set(title=f"Cancellation rate by {col}", xlabel="cancellation rate (%)", ylabel="")

plt.tight_layout()
plt.show()


**Observation.** `deposit_type` stands out as one of the stronger features. Interestingly, `Non Refund` bookings have a much higher cancellation rate than `No Deposit` bookings, which is the opposite of what we would normally expect. This seems to be a quirk of this dataset, so the model may pick up on this pattern even though it may not apply to hotels in general. `market_segment` also shows a higher cancellation rate for `Groups` compared to `Direct` and `Corporate`, while `customer_type` shows that `Transient` guests cancel more often than `Group` and `Contract` guests.

## 3. Cleaning  *(B1)*

### 3.1 Columns that cannot be used

The prediction moment is fixed by the problem statement: **the instant the
booking is made**. Any column only settled after that point is a leak,
however predictive.

In [ ]:
LEAKY = {
    "reservation_status":      "check-out / cancelled / no-show - this *is* the outcome",
    "reservation_status_date": "the date that status was set - same information",
    "assigned_room_type":      "the room actually given, decided at check-in",
}

print("reservation_status vs is_canceled - why it has to go:")
print(pd.crosstab(df_raw["reservation_status"], df_raw["is_canceled"]))

df = df_raw.drop(columns=list(LEAKY))
for col, why in LEAKY.items():
    print(f"\ndropped  {col:24s}  {why}")
print(f"\ncolumns: {df_raw.shape[1]} -> {df.shape[1]}")


**Observation.** `reservation_status` maps onto `is_canceled` almost
one-to-one - every `Canceled` row is `is_canceled = 1` and every `Check-Out`
row is `is_canceled = 0` (`No-Show` also counts as cancelled here). Training
on it would give near-perfect accuracy for a reason that is useless in
practice: it is the answer, recorded after the fact, not a predictor.
`assigned_room_type` is tempting because it correlates with cancellation
(mismatched assignments are rarer for guests who never show up to notice),
but it is only known at check-in, so it is dropped for the same reason as in
the regression notebook.

### 3.2 Missing values - four columns, four treatments

In [ ]:
# company: 94.3% missing. The id itself is unusable, but whether a corporate
# account was attached is a genuine signal, so keep the flag and drop the id.
df["has_company"] = df["company"].notna().astype(int)
df = df.drop(columns=["company"])

# agent: a blank means "booked without a travel agent" - informative absence.
# Keep it as a category with an explicit level rather than imputing an id.
df["has_agent"] = df["agent"].notna().astype(int)
df["agent"] = df["agent"].fillna(0).astype(int).astype(str)

# children: 4 rows, and a missing child count means no children were declared.
df["children"] = df["children"].fillna(0)

# country: 0.4% genuinely unknown - gets its own level, not a mode fill, so
# the model can learn whether an unknown origin behaves differently.
df["country"] = df["country"].fillna("Unknown")

# meal: "Undefined" and "SC" both mean no meal package (Antonio et al. 2019).
df["meal"] = df["meal"].replace({"Undefined": "SC"})

print(f"remaining missing values: {df.isna().sum().sum()}")
print(f"\nhas_company = 1 : {df['has_company'].mean() * 100:5.2f}%")
print(f"has_agent   = 1 : {df['has_agent'].mean() * 100:5.2f}%")
print(f"meal levels     : {sorted(df['meal'].unique())}")


**Observation.** Same four columns as the regression track need
attention, and the same reasoning applies since it is a property of the raw
data rather than the target: nothing gets a mean or mode fill. Three gaps
are structural absences worth keeping as flags, and only `children` gets a
constant fill, justified by it being 4 rows out of 119,390.

### 3.3 Invalid rows

In [ ]:
rows = [("raw", len(df))]

df = df[(df["adults"] + df["children"] + df["babies"]) > 0]
rows.append(("after zero-guest filter", len(df)))

df = df[df["adr"] >= 0]
rows.append(("after negative-ADR filter", len(df)))

df = df.drop_duplicates().reset_index(drop=True)
rows.append(("after de-duplication", len(df)))

log = pd.DataFrame(rows, columns=["stage", "rows"])
log["removed"] = -log["rows"].diff().fillna(0).astype(int)
log["% of raw"] = (log["rows"] / rows[0][1] * 100).round(1)
log


In [ ]:
print(f"clean dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\ncancellation rate after cleaning: {df['is_canceled'].mean() * 100:.1f}%")


**Observation.** Zero-guest rows (180) are a data-entry error rather
than a real booking, the single negative-ADR row is the same. Duplicates take
the largest bite at roughly 28% of rows. Unlike the regression track, this
one is not neutral: the cancellation rate drops from 37.0% before cleaning to
27.6% after, because duplicated rows cancel at 57% against 26% for
non-duplicated rows - large identical-looking blocks (group and online-TA
bookings, per Section 2.5) are exactly the ones over-represented by exact
duplication. Dropping them is still correct, since a repeated row would
otherwise be counted - and could land in both train and test - multiple
times, but it does shift the class balance the models are trained on.

## 4. Feature engineering  *(B3)*

In [ ]:
MONTHS = ["January", "February", "March", "April", "May", "June", "July",
          "August", "September", "October", "November", "December"]

# Party composition - Section 2.5 showed engagement signals (special
# requests) matter more than raw numbers, but who is in the booking still
# shapes how committed it is.
df["total_guests"] = df["adults"] + df["children"] + df["babies"]
df["is_family"]    = ((df["children"] + df["babies"]) > 0).astype(int)

# Stay length - a one-night booking and a two-week booking carry different
# cancellation risk even at the same lead time.
df["total_nights"]  = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["weekend_ratio"] = df["stays_in_weekend_nights"] / df["total_nights"].replace(0, np.nan)
df["weekend_ratio"] = df["weekend_ratio"].fillna(0)

# Seasonality - week 52 and week 1 are adjacent in the calendar but 51 apart
# on a number line. Sine/cosine restores that adjacency for non-tree models.
df["month_num"] = df["arrival_date_month"].map({m: i + 1 for i, m in enumerate(MONTHS)})
df["week_sin"]  = np.sin(2 * np.pi * df["arrival_date_week_number"] / 52)
df["week_cos"]  = np.cos(2 * np.pi * df["arrival_date_week_number"] / 52)

# Guest history - the single strongest engineered signal for this target.
# previous_cancellations alone (Section 2.3) barely correlates with the
# outcome because it does not distinguish a guest with 1 cancellation out of
# 1 stay from one with 1 cancellation out of 20. The rate does.
df["prev_total"]       = df["previous_cancellations"] + df["previous_bookings_not_canceled"]
df["prev_cancel_rate"] = df["previous_cancellations"] / (df["prev_total"] + 1)

df = df.drop(columns=["arrival_date_month"])

NEW = ["total_guests", "is_family", "total_nights", "weekend_ratio",
       "month_num", "week_sin", "week_cos", "prev_total", "prev_cancel_rate",
       "has_agent", "has_company"]
print(f"{len(NEW)} engineered features\n")
df[NEW].describe().T.round(3)


In [ ]:
engineered_corr = (df[NEW + ["is_canceled"]].corr()["is_canceled"]
                   .drop("is_canceled").sort_values(key=abs, ascending=False))
print("Correlation with is_canceled:")
print(engineered_corr.round(3))


**Observation.** `prev_cancel_rate` is the strongest engineered
feature (r = 0.160) - it captures something no raw column does directly: a
guest's own track record. `has_agent` is a close second (r = 0.134),
consistent with the deposit/agency pattern already visible in Section 2.5.
`weekend_ratio` and `week_sin` are essentially flat (r < 0.01); party
composition and calendar position matter far less here than they did for
`adr` in the regression track. These engineered columns are not expected to
dominate individually - tree-based models in Part B will combine them with
the raw columns in ways a single correlation coefficient cannot show.

## 5. Encoding, scaling & splitting  *(B2)*

### 5.1 High-cardinality categoricals

Same issue as the regression track: `country` has 178 levels and `agent`
334. One-hot encoding them as-is would add hundreds of sparse columns, most
of them near-constant zero for any given model. Grouping the long tail into
`Other` keeps the useful levels and drops the noise.

In [ ]:
TOP_N = {"country": 20, "agent": 15}

for col, n in TOP_N.items():
    top = df[col].value_counts().nlargest(n).index
    covered = df[col].isin(top).mean() * 100
    print(f"{col:8s}: {df[col].nunique():4d} levels -> top {n} cover {covered:.1f}% of rows")
    df[col] = np.where(df[col].isin(top), df[col], "Other")

print()
print(df[["country", "agent"]].nunique().to_string())


### 5.2 Preprocessing pipeline and split

Every transform lives inside a `ColumnTransformer` fit only on the training
split, then applied to both splits - fitting a scaler or encoder on the full
dataset before splitting would leak test-set statistics into training, an
explicit data-leakage error under the course guidelines. The split is
stratified on `is_canceled` itself, which is the natural choice for a
classification target (no banding trick needed, unlike the continuous `adr`
target in the regression notebook).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

y = df["is_canceled"]
X = df.drop(columns=["is_canceled"])

CATEGORICAL = X.select_dtypes(include=["object", "str"]).columns.tolist()
NUMERIC = [c for c in X.columns if c not in CATEGORICAL]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
    ("num", StandardScaler(), NUMERIC),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

n_features = preprocessor.fit_transform(X_train).shape[1]
print(f"categorical columns : {len(CATEGORICAL):2d}  {CATEGORICAL}")
print(f"numeric columns     : {len(NUMERIC):2d}")
print(f"after encoding      : {n_features} model features")
print(f"\ntrain: {X_train.shape[0]:,} rows    test: {X_test.shape[0]:,} rows")
print(f"train cancellation rate {y_train.mean()*100:.1f}%    test cancellation rate {y_test.mean()*100:.1f}%")


**Observation.** Stratifying on `is_canceled` keeps the train and test
cancellation rates within a few tenths of a percent of each other, so the
80:20 split is not accidentally handing one side a harder or easier problem.
StandardScaler is applied for every model, including the tree-based Decision
Tree, purely for pipeline consistency across the five algorithms - a
monotonic rescaling does not change where a tree splits, so it costs
nothing there while being required for Logistic Regression, KNN and SVM.